# Encodage Catégoriel
Entrée : `metadata_features.parquet` → Sortie : `features_encodees.parquet`

- Suppression des colonnes string constantes, redondantes ou à trop haute cardinalité
- One-hot encoding des colonnes catégorielles utiles (drop_first=True)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'

df = pd.read_parquet(DATA_PROCESSED / 'metadata_features.parquet')
df_enc = df.copy()

print('Dataset chargé :', df_enc.shape)

Dataset chargé : (549971, 395)


## 1. Suppression des colonnes string inutiles

In [2]:
DROP_REDUNDANT = [
    'in.building_america_climate_zone',
    'in.cec_climate_zone',
    'in.energystar_climate_zone_2023',
    # 'in.ashrae_iecc_climate_zone_2004_sub_cz_split',  # conserve -> decompose en climate_severity / moisture / 2A_west
    'in.hvac_heating_type_and_fuel',
    'in.census_division',
    'in.census_division_recs',
    'in.geometry_building_type_acs',
    'in.location_region',
    'in.ahs_region',
    'in.iso_rto_region',
    'in.puma_metro_status',
    'in.county_metro_status',
    'in.generation_and_emissions_assessment_region',
    'in.metropolitan_and_micropolitan_statistical_area',
    'in.reeds_balancing_area',
    'in.county_and_puma',
    'in.county_name',
    'in.county',
    'in.puma',
    'in.city',
    'in.state',
    'in.weather_file_city',
    'in.heating_unavailable_period',
    'in.cooling_unavailable_period',
    'in.upgrade_name',
    'in.holiday_lighting',
    'in.hot_water_distribution',
    # Colonnes string deja decomposees en numeriques dans transformations_numeriques
    'in.duct_leakage_and_insulation',  # -> in.duct_leakage + in.duct_insulation
    'in.duct_location',                # -> in.duct_location_int
    'in.ground_thermal_conductivity',  # CSV: Supprimer (peu discriminant)
]
str_cols = df_enc.select_dtypes(exclude=[np.number]).columns
drop_const    = [c for c in str_cols if df_enc[c].nunique() <= 1]
drop_explicit = [c for c in DROP_REDUNDANT if c in df_enc.columns]

to_drop = list(set(drop_const + drop_explicit))
df_enc.drop(columns=to_drop, inplace=True)

print(f'Colonnes constantes supprimees  : {len(drop_const)}')
print(f'Colonnes redondantes supprimees : {len(drop_explicit)}')
print(f'Shape : {df_enc.shape}')


Colonnes constantes supprimees  : 21
Colonnes redondantes supprimees : 30
Shape : (549971, 347)


## 2. Imputation par groupe (zone climatique)
Doit s'executer ici — avant le OHE — pendant que la zone climatique est encore une chaine.
Utilise  (plus granulaire que ASHRAE classique).

In [3]:
GROUPBY_COL = 'in.ashrae_iecc_climate_zone_2004_sub_cz_split'

# Fallback sur ashrae classique si sub_cz_split absent
if GROUPBY_COL not in df_enc.columns:
    GROUPBY_COL = 'in.ashrae_iecc_climate_zone_2004'

num_cols = df_enc.select_dtypes(include=[np.number]).columns
nan_cols = [c for c in num_cols if df_enc[c].isna().any()]

if nan_cols and GROUPBY_COL in df_enc.columns:
    for c in nan_cols:
        global_med  = df_enc[c].median()
        group_meds  = df_enc.groupby(GROUPBY_COL)[c].transform('median')
        df_enc[c]   = df_enc[c].fillna(group_meds.fillna(global_med))
    print(f'Imputation par zone climatique ({GROUPBY_COL}) : {len(nan_cols)} colonnes')
    print(f'NaN residuels : {df_enc[nan_cols].isna().sum().sum()}')
else:
    print('Colonne de groupement absente - ignore.')


Imputation par zone climatique (in.ashrae_iecc_climate_zone_2004_sub_cz_split) : 29 colonnes
NaN residuels : 0


## 3. Groupe 1 - Zone climatique (encodage structure)

Remplacement du OHE brut sur `in.ashrae_iecc_climate_zone_2004` (17 dummies) par 3 features semantiques :

| Feature creee | Description |
|---|---|
| `climate_severity` | Chiffre de la zone (1->8), **ordinal** : severite thermique croissante |
| `moisture_B` / `moisture_C` / `moisture_AK` | Lettre de la zone (**OHE**) : A=humide (ref), B=sec, C=marin, AK=Alaska |
| `climate_2A_west` | **Binaire** : 1 si zone 2A TX/LA (vs FL/GA/AL/MS) - comportement HVAC distinct |

Source : `in.ashrae_iecc_climate_zone_2004_sub_cz_split` (superset de ASHRAE classique)

In [4]:
col_split  = 'in.ashrae_iecc_climate_zone_2004_sub_cz_split'
col_ashrae = 'in.ashrae_iecc_climate_zone_2004'

# Utiliser sub_cz_split si disponible, sinon fallback ashrae classique
col = col_split if col_split in df_enc.columns else col_ashrae

# 1. Severite thermique : chiffre (1-8), ordinal
df_enc['climate_severity'] = df_enc[col].str.extract(r'^(\d+)').astype(float).astype('float32')

# 2. Type d'humidite : lettre (A/B/C), zones AK -> categorie propre
moisture_raw = df_enc[col].str.extract(r'^\d+([A-C])')[0]
ak_mask      = df_enc[col].str.contains('AK', na=False)
moisture_raw = moisture_raw.where(~ak_mask, other='AK')
df_enc['climate_moisture'] = moisture_raw

df_enc = pd.get_dummies(df_enc, columns=['climate_moisture'],
                         prefix='moisture', dtype=np.int8, drop_first=True)

# 3. Binaire 2A west : TX/LA vs FL/GA/AL/MS (comportement HVAC distinct)
df_enc['climate_2A_west'] = df_enc[col].str.contains('TX|LA', na=False).astype(np.int8)

# Supprimer les deux colonnes source
for c in [col_ashrae, col_split]:
    if c in df_enc.columns:
        df_enc.drop(columns=[c], inplace=True)

print(f'climate_severity   : {df_enc["climate_severity"].value_counts().sort_index().to_dict()}')
moisture_cols = [c for c in df_enc.columns if c.startswith('moisture_')]
print(f'moisture columns   : {moisture_cols}')
print(f'climate_2A_west    : {df_enc["climate_2A_west"].value_counts().to_dict()}')
print(f'Shape : {df_enc.shape}')


climate_severity   : {1.0: 9855, 2.0: 74622, 3.0: 137172, 4.0: 138508, 5.0: 144793, 6.0: 38801, 7.0: 5934, 8.0: 286}
moisture columns   : ['moisture_AK', 'moisture_B', 'moisture_C']
climate_2A_west    : {0: 520227, 1: 29744}
Shape : (549971, 350)


## 4. Groupe 2 — Géométrie

| Colonne | Stratégie |
|---|---|
| `in.orientation` | sin/cos circulaire (8 directions) |
| `in.window_areas` | extraction numérique F/B/L/R (%) |
| `in.geometry_building_number_units_mf/sfa` | midpoint numérique |
| `in.geometry_building_level_mf` | ordinal (Bottom=0, Middle=1, Top=2) |
| `in.geometry_building_horizontal_location_mf/sfa` | OHE drop_first |
| `in.geometry_attic_type` | ordinal thermique (Not Applicable=0 → Conditioned=3) |
| `in.geometry_foundation_type` | ordinal thermique (Ambient=0 → Heated Basement=4) |
| `in.geometry_garage` | parse → `garage_cars` (0-3) + `garage_conditioned` (0/1) |
| `in.neighbors` | ordinal (None=0, un côté=1, Both=2) |
| `in.door_area` | numérique ft² → m² |
| `in.corridor` | supprimé (tout NaN après transformations_numeriques) |
| `in.geometry_wall_type` | OHE drop_first |
| `in.geometry_wall_exterior_finish` | OHE drop_first |
| `in.geometry_building_type_recs` | OHE drop_first |

In [5]:
import re

# Orientation -> sin/cos
ORIENT_DEG = {
    'North': 0, 'Northeast': 45, 'East': 90, 'Southeast': 135,
    'South': 180, 'Southwest': 225, 'West': 270, 'Northwest': 315,
}
col = 'in.orientation'
if col in df_enc.columns:
    deg = df_enc[col].map(ORIENT_DEG)
    df_enc['in.orientation_sin'] = np.sin(np.radians(deg)).astype('float32')
    df_enc['in.orientation_cos'] = np.cos(np.radians(deg)).astype('float32')
    df_enc.drop(columns=[col], inplace=True)
    print('in.orientation → sin/cos OK')

# ── Window areas → F/B/L/R numériques ───────────────────────────────────────
def parse_window_areas(val):
    if pd.isna(val):
        return np.nan, np.nan, np.nan, np.nan
    parts = re.findall(r'([FBLR])(\d+)', str(val).upper())
    d = {k: float(v) for k, v in parts}
    return d.get('F', np.nan), d.get('B', np.nan), d.get('L', np.nan), d.get('R', np.nan)

col = 'in.window_areas'
if col in df_enc.columns:
    parsed = df_enc[col].apply(lambda v: pd.Series(parse_window_areas(v),
                                index=['in.window_front', 'in.window_back', 'in.window_left', 'in.window_right']))
    df_enc = pd.concat([df_enc.drop(columns=[col]), parsed.astype('float32')], axis=1)
    print('in.window_areas → 4 colonnes OK')

# ── Number of units → colonne fusionnée mf + sfa ────────────────────────────
def midpoint_units(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s in ('None', 'nan'): return np.nan
    if s.endswith('+'): return float(s[:-1])
    m = re.match(r'(\d+)-(\d+)', s)
    if m: return (float(m.group(1)) + float(m.group(2))) / 2
    return pd.to_numeric(s, errors='coerce')

col_mf  = 'in.geometry_building_number_units_mf'
col_sfa = 'in.geometry_building_number_units_sfa'
if col_mf in df_enc.columns or col_sfa in df_enc.columns:
    mf  = df_enc[col_mf].apply(midpoint_units)  if col_mf  in df_enc.columns else pd.Series(np.nan, index=df_enc.index)
    sfa = df_enc[col_sfa].apply(midpoint_units) if col_sfa in df_enc.columns else pd.Series(np.nan, index=df_enc.index)
    df_enc['in.building_number_units'] = mf.combine_first(sfa).astype('float32')
    df_enc.drop(columns=[c for c in [col_mf, col_sfa] if c in df_enc.columns], inplace=True)
    print('in.building_number_units (mf + sfa fusionnés) OK')

# ── Building level → ordinal ─────────────────────────────────────────────────
LEVEL_MAP = {'Bottom': 0, 'Middle': 1, 'Top': 2}
col = 'in.geometry_building_level_mf'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map(LEVEL_MAP).astype('Int8')
    print('in.geometry_building_level_mf → ordinal OK')

# ── Horizontal location → colonne fusionnée mf + sfa → OHE ──────────────────
col_mf  = 'in.geometry_building_horizontal_location_mf'
col_sfa = 'in.geometry_building_horizontal_location_sfa'
if col_mf in df_enc.columns or col_sfa in df_enc.columns:
    loc_mf  = df_enc[col_mf].replace({'None': np.nan, 'Not Applicable': np.nan}) if col_mf  in df_enc.columns else pd.Series(np.nan, index=df_enc.index)
    loc_sfa = df_enc[col_sfa].replace({'None': np.nan})                           if col_sfa in df_enc.columns else pd.Series(np.nan, index=df_enc.index)
    df_enc['in.building_horizontal_location'] = loc_mf.combine_first(loc_sfa)
    df_enc.drop(columns=[c for c in [col_mf, col_sfa] if c in df_enc.columns], inplace=True)
    df_enc['in.building_horizontal_location'] = df_enc['in.building_horizontal_location'].astype('category')
    df_enc = pd.get_dummies(df_enc, columns=['in.building_horizontal_location'],
                            prefix='in.horiz_loc', dtype=np.int8, drop_first=True)
    print('in.building_horizontal_location (mf + sfa fusionnés) → OHE OK')

print(f'\nShape : {df_enc.shape}')

in.orientation → sin/cos OK


in.window_areas → 4 colonnes OK


in.building_number_units (mf + sfa fusionnés) OK
in.geometry_building_level_mf → ordinal OK
in.building_horizontal_location (mf + sfa fusionnés) → OHE OK

Shape : (549971, 353)


In [6]:
import re

# ── sqft..ft2 → suppression (redondant avec geometry_floor_area) ─────────────
if 'in.sqft..ft2' in df_enc.columns:
    df_enc.drop(columns=['in.sqft..ft2'], inplace=True)
    print('in.sqft..ft2 supprimé (redondant avec geometry_floor_area)')

# ── door_area → suppression (colonne constante : "20 ft^2") ──────────────────
if 'in.door_area' in df_enc.columns:
    df_enc.drop(columns=['in.door_area'], inplace=True)
    print('in.door_area supprimé (constante)')

# ── Attic type → ordinal (qualité thermique croissante) ─────────────────────
ATTIC_MAP = {
    'None':                                 0,  # pas de combles
    'Vented Attic':                         1,  # combles ventilés (non conditionnés)
    'Unvented Attic':                       2,  # combles non ventilés (mieux isolés)
    'Finished Attic or Cathedral Ceilings': 3,  # combles aménagés/conditionnés
}
col = 'in.geometry_attic_type'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map(ATTIC_MAP).astype('Int8')
    print(f'attic_type → ordinal OK  {df_enc[col].value_counts().sort_index().to_dict()}')

# ── Foundation type → ordinal (gradient thermique sol) ──────────────────────
FOUNDATION_MAP = {
    'Ambient':             0,
    'Slab':                1,
    'Vented Crawlspace':   2,   # ouvert sur l air exterieur -> plancher plus froid
    'Unvented Crawlspace': 3,   # ferme et isole -> plancher moins froid
    'Unheated Basement':   4,
    'Heated Basement':     5,
}
col = 'in.geometry_foundation_type'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map(FOUNDATION_MAP).astype('Int8')
    print(f'foundation_type → ordinal OK  {df_enc[col].value_counts().sort_index().to_dict()}')

# ── Garage → ordinal (nombre de voitures) ───────────────────────────────────
GARAGE_MAP = {'None': 0, '1 Car': 1, '2 Car': 2, '3 Car': 3}
col = 'in.geometry_garage'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map(GARAGE_MAP).astype('Int8')
    print(f'garage → ordinal OK  {df_enc[col].value_counts().sort_index().to_dict()}')

# ── Neighbors → distance (ft) + flag both sides ──────────────────────────────
def parse_neighbors(val):
    if pd.isna(val): return np.nan, 0
    s = str(val).strip()
    if s == 'None': return 0.0, 0
    m = re.search(r'(\d+)', s)
    dist = float(m.group(1)) if m else np.nan
    both = int('Left/Right' in s or 'Both' in s)
    return dist, both

col = 'in.neighbors'
if col in df_enc.columns:
    parsed = df_enc[col].apply(
        lambda v: pd.Series(parse_neighbors(v), index=['in.neighbor_distance_ft', 'in.neighbor_both_sides'])
    )
    df_enc = pd.concat([df_enc.drop(columns=[col]), parsed.astype('float32')], axis=1)
    print('in.neighbors → in.neighbor_distance_ft + in.neighbor_both_sides OK')

# ── Corridor → binaire ───────────────────────────────────────────────────────
CORRIDOR_MAP = {'Not Applicable': 0, 'Double-Loaded Interior': 1}
col = 'in.corridor'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map(CORRIDOR_MAP).fillna(0).astype('Int8')
    print(f'in.corridor → binaire OK  {df_enc[col].value_counts().to_dict()}')

# ── Wall exterior finish → matériau (OHE) + réflectivité (binaire) ───────────
FINISH_MATERIAL = {
    'Vinyl, Light':                 'vinyl',
    'Aluminum, Light':              'aluminum',
    'Wood, Medium/Dark':            'wood',
    'Brick, Light':                 'brick',
    'Brick, Medium/Dark':           'brick',
    'Stucco, Light':                'stucco',
    'Stucco, Medium/Dark':          'stucco',
    'Fiber-Cement, Light':          'fiber_cement',
    'Shingle, Composition, Medium': 'shingle',
    'Shingle, Asbestos, Medium':    'shingle',
    'None':                         'none',
}
col = 'in.geometry_wall_exterior_finish'
if col in df_enc.columns:
    EXTERIOR_FINISH_R = {
        'Aluminum, Light':              0.6,
        'Brick, Light':                 0.7,
        'Brick, Medium/Dark':           0.7,
        'Fiber-Cement, Light':          0.2,
        'None':                         0.0,
        'Shingle, Asbestos, Medium':    0.6,
        'Shingle, Composition, Medium': 0.6,
        'Stucco, Light':                0.2,
        'Stucco, Medium/Dark':          0.2,
        'Vinyl, Light':                 0.6,
        'Wood, Medium/Dark':            1.4,
    }
    df_enc['in.wall_exterior_finish_r'] = df_enc[col].map(EXTERIOR_FINISH_R).astype('float32')
    df_enc['in.wall_finish_material'] = df_enc[col].map(FINISH_MATERIAL).astype('category')
    df_enc['in.wall_finish_dark']     = df_enc[col].str.contains('Medium|Dark', na=False).astype(np.int8)
    df_enc.drop(columns=[col], inplace=True)
    df_enc = pd.get_dummies(df_enc, columns=['in.wall_finish_material'],
                            prefix='in.wall_finish', dtype=np.int8, drop_first=True)
    print('in.wall_exterior_finish → in.wall_finish_<materiau> + in.wall_finish_dark OK')

# ── Wall type + building type RECS → OHE ─────────────────────────────────────
OHE_COLS = {
    'in.geometry_wall_type':          'wall_type',
    'in.geometry_building_type_recs': 'bldg_type',
}
ohe_present = {k: v for k, v in OHE_COLS.items() if k in df_enc.columns}
if ohe_present:
    for c in ohe_present:
        df_enc[c] = df_enc[c].astype('category')
    df_enc = pd.get_dummies(df_enc, columns=list(ohe_present.keys()),
                            prefix=list(ohe_present.values()),
                            dtype=np.int8, drop_first=True)
    print(f'OHE : {list(ohe_present.keys())}')

print(f'\nShape : {df_enc.shape}')

in.sqft..ft2 supprimé (redondant avec geometry_floor_area)
attic_type → ordinal OK  {np.int8(0): 306076, np.int8(1): 217772, np.int8(2): 18223, np.int8(3): 7900}


foundation_type → ordinal OK  {np.int8(0): 48793, np.int8(1): 216490, np.int8(2): 121700, np.int8(3): 10416, np.int8(4): 84384, np.int8(5): 68188}
garage → ordinal OK  {np.int8(0): 371486, np.int8(1): 57157, np.int8(2): 106887, np.int8(3): 14441}


in.neighbors → in.neighbor_distance_ft + in.neighbor_both_sides OK
in.corridor → binaire OK  {np.int8(0): 404668, np.int8(1): 145303}
in.wall_exterior_finish → in.wall_finish_<materiau> + in.wall_finish_dark OK


OHE : ['in.geometry_wall_type', 'in.geometry_building_type_recs']

Shape : (549971, 366)


## 5. Groupe 3 — Enveloppe thermique

| Colonne | Stratégie |
|---|---|
| `in.insulation_wall/ceiling/roof/floor/foundation_wall/rim_joist/slab` | déjà R-values numériques (`transformations_numeriques`) |
| `in.air_leakage_to_outside_ach50` | déjà float |
| `in.infiltration` | supprimé (redondant avec `air_leakage_to_outside_ach50`) |
| `in.interior_shading` / `in.overhangs` / `in.eaves` / `in.doors` / `in.radiant_barrier` | supprimés (colonnes constantes) |
| `in.windows` | décomposé → `window_panes` (1/2/3) + `in.window_low_e` + `in.window_metal_frame` + `in.window_storm` |
| `in.ground_thermal_conductivity` | numérique float (W/m·K) |
| `in.roof_material` | à traiter (collègue) |

In [7]:
# Colonnes constantes + redondantes a supprimer
DROP_ENVELOPE = [
    'in.interior_shading',
    'in.overhangs',
    'in.eaves',
    'in.doors',
    'in.radiant_barrier',
    'in.infiltration',
]
dropped = [c for c in DROP_ENVELOPE if c in df_enc.columns]
df_enc.drop(columns=dropped, inplace=True)
print(f'Supprimes ({len(dropped)}) : {dropped}')

# Windows : U-factor + SHGC (Table 69 ResStock guide)
# U-factor : transmittance thermique, plus bas = mieux isole
# SHGC     : gain solaire, plus bas = moins de chaleur solaire
WINDOWS_PARAMS = {
    'Single, Clear, Metal':                                (1.16, 0.76),
    'Single, Clear, Metal, Exterior Clear Storm':          (0.67, 0.56),
    'Single, Clear, Non-metal':                            (0.84, 0.63),
    'Single, Clear, Non-metal, Exterior Clear Storm':      (0.47, 0.54),
    'Double, Clear, Metal, Air':                           (0.76, 0.67),
    'Double, Clear, Metal, Air, Exterior Clear Storm':     (0.55, 0.51),
    'Double, Clear, Non-metal, Air':                       (0.49, 0.56),
    'Double, Clear, Non-metal, Air, Exterior Clear Storm': (0.34, 0.49),
    'Double, Low-E, Non-metal, Air, M-Gain':               (0.38, 0.44),
    'Triple, Low-E, Non-metal, Air, L-Gain':               (0.29, 0.26),
}
col = 'in.windows'
if col in df_enc.columns:
    parsed = df_enc[col].apply(
        lambda v: pd.Series(
            WINDOWS_PARAMS.get(str(v), (np.nan, np.nan)),
            index=['in.window_ufactor', 'in.window_shgc']
        )
    )
    df_enc = pd.concat([df_enc.drop(columns=[col]), parsed.astype('float32')], axis=1)
    print('in.windows -> in.window_ufactor + in.window_shgc OK')

# Ground thermal conductivity
col = 'in.ground_thermal_conductivity'
if col in df_enc.columns:
    df_enc[col] = pd.to_numeric(df_enc[col], errors='coerce').astype('float32')
    print(f'ground_thermal_conductivity -> float OK')

print(f'Shape : {df_enc.shape}')

Supprimes (2) : ['in.radiant_barrier', 'in.infiltration']


in.windows -> in.window_ufactor + in.window_shgc OK
Shape : (549971, 365)


## 6. Groupe 3 (suite) — Roof material
Encodage par R-value thermique (isolation thermique du matériau de toiture)

In [8]:
#https://roofobservations.com/r-value-table-building-materials/#google_vignette
#https://www.cupapizarras.com/uk/news/thermal-behavior-roofing-slates/
#https://web.ornl.gov/sci/buildings/conf-archive/2013%20B12%20papers/163-Olson.pdf
#https://ruggedcoatings.com/understanding-r-value-in-roofing-systems/
#https://lorisweb.com/CMGT235/DIS02/R-Values%20Lookup.pdf

def roof_type(rf_str):
    if pd.isna(rf_str):
        return np.nan
    s = str(rf_str).strip()
    if 'Asphalt Shingles, Medium' in s:
        return 0.44
    elif 'Composition Shingles' in s:
        return 0.44
    elif 'Metal, Dark' in s:
        return 0.17
    elif 'Slate' in s:
        return 0.05
    elif 'Tile, Clay or Ceramic' in s:
        return 0.80
    elif 'Tile, Concrete' in s:
        return 0.60
    elif 'Wood Shingles' in s:
        return 1.00
    return np.nan

col = 'in.roof_material'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].apply(roof_type).astype('float32')
    print(f'{col} → R-value OK',df_enc[col].value_counts().sort_index().to_dict())
print(f'\nShape : {df_enc.shape}')    

in.roof_material → R-value OK {0.05000000074505806: 6690, 0.17000000178813934: 46281, 0.4399999976158142: 443356, 0.6000000238418579: 9344, 0.800000011920929: 21130, 1.0: 23170}

Shape : (549971, 365)


## 5. Groupe 3 — Systèmes HVAC
`in.hvac_heating_type`, `in.hvac_cooling_type`, `in.hvac_heating_fuel` — OHE drop_first

In [9]:
HVAC_COLS = [
    'in.hvac_heating_type',
    'in.hvac_cooling_type',
    'in.hvac_heating_fuel',
]

hvac_cols_present = [c for c in HVAC_COLS if c in df_enc.columns]

for c in hvac_cols_present:
    df_enc[c] = df_enc[c].astype('category')

df_enc = pd.get_dummies(df_enc, columns=hvac_cols_present,
                        prefix=[c.split('in.')[-1] for c in hvac_cols_present],
                        dtype=np.int8, drop_first=True)

print(f'Colonnes HVAC encodées : {hvac_cols_present}')
print(f'Shape : {df_enc.shape}')

Colonnes HVAC encodées : ['in.hvac_heating_type', 'in.hvac_cooling_type']
Shape : (549971, 371)


In [10]:
# ── Heating efficiency → séparer AFUE (fuel/élec) et HSPF (PAC) ──────────────
# Problème : la colonne mélange deux métriques incomparables
#   AFUE ∈ [0.60, 1.0]  → combustible (gaz, fuel) + électrique résistance
#   HSPF ∈ [6.2,  8.5]  → pompes à chaleur air-air (ASHP)
# Séparation par seuil : HSPF > 1.0, AFUE ≤ 1.0
# AFUE = 1.0 (électrique résistance) → NaN : constant par type, déjà encodé dans hvac_heating_type

col = 'in.hvac_heating_efficiency'
if col in df_enc.columns:
    eff = df_enc[col]
    df_enc['in.hvac_heating_eff_hspf'] = eff.where(eff > 1.0).astype('float32')
    df_enc['in.hvac_heating_eff_afue'] = eff.where(eff < 1.0).astype('float32')
    df_enc.drop(columns=[col], inplace=True)
    print('hvac_heating_efficiency → 2 colonnes :')
    print(f'  in.hvac_heating_eff_afue : {sorted(df_enc["in.hvac_heating_eff_afue"].dropna().unique())}')
    print(f'  in.hvac_heating_eff_hspf : {sorted(df_enc["in.hvac_heating_eff_hspf"].dropna().unique())}')

# ── Cooling efficiency → SEER/EER, même ordre de grandeur → une seule colonne ─
# NaN pour les PAC (leur efficacité chauffage est dans hvac_heating_eff_hspf)
col = 'in.hvac_cooling_efficiency'
if col in df_enc.columns:
    df_enc[col] = pd.to_numeric(df_enc[col], errors='coerce').astype('float32')
    print(f'in.hvac_cooling_efficiency → SEER/EER float OK  {sorted(df_enc[col].dropna().unique())}')

print(f'\nShape : {df_enc.shape}')

hvac_heating_efficiency → 2 colonnes :
  in.hvac_heating_eff_afue : [np.float32(0.6), np.float32(0.68), np.float32(0.76), np.float32(0.8), np.float32(0.9), np.float32(0.925)]
  in.hvac_heating_eff_hspf : [np.float32(6.2), np.float32(7.7), np.float32(8.2), np.float32(8.5), np.float32(14.0)]
in.hvac_cooling_efficiency → SEER/EER float OK  [np.float32(8.0), np.float32(8.5), np.float32(9.8), np.float32(10.0), np.float32(10.7), np.float32(12.0), np.float32(13.0), np.float32(15.0)]

Shape : (549971, 372)


## 6. Groupe 4 - Periode d offset consignes (encodage corrige)

Encodage separe des composantes **Day** et **Night** avec leurs heures de debut correctes.

| Feature | Description |
|---|---|
| heat/cool_period_type | 0=None, 1=Day, 2=Night, 3=Both |
| heat/cool_day_start_sin/cos | sin/cos heure debut jour (9h + shift) |
| heat/cool_night_start_sin/cos | sin/cos heure debut nuit (22h + shift) |
| cool_day_direction | +1=Setup, -1=Setback, composante jour uniquement |
| cool_night_direction | +1=Setup, -1=Setback, composante nuit uniquement |

**Correction vs ancienne version** :
- Ancienne : un seul default_start pour Day ET Night -> sin/cos faux pour Day en chauffage, Night en clim
- Ancienne : 'Day Setup and Night Setback' -> direction = 1-1 = 0, information perdue
- Nouvelle : chaque composante a sa propre heure et direction

In [11]:
import re
import numpy as np

DAY_START   = 9
NIGHT_START = 22

def encode_offset_period_v2(val, with_direction=True):
    """
    Encode offset period avec composantes Day/Night separees et heures correctes.
    - Day  : debut a DAY_START   (9h) + shift
    - Night: debut a NIGHT_START (22h) + shift
    - Direction separee par composante (pour cooling: Day Setup vs Night Setback)
    """
    zeros = {
        'period_type':     0,
        'day_start_sin':   0., 'day_start_cos':   1.,
        'night_start_sin': 0., 'night_start_cos': 1.,
    }
    if with_direction:
        zeros.update({'day_direction': 0, 'night_direction': 0})

    if pd.isna(val) or str(val).strip().lower() == 'none':
        return zeros

    s = str(val).strip().lower()
    m = re.search(r'([+-]\d+)h', s)
    shift = int(m.group(1)) if m else 0

    has_day   = 'day'   in s
    has_night = 'night' in s
    period_type = int(has_day) + 2 * int(has_night)

    day_h   = (DAY_START   + shift) % 24
    night_h = (NIGHT_START + shift) % 24

    result = {
        'period_type':     period_type,
        'day_start_sin':   float(np.sin(2*np.pi*day_h  /24)) if has_day   else 0.,
        'day_start_cos':   float(np.cos(2*np.pi*day_h  /24)) if has_day   else 1.,
        'night_start_sin': float(np.sin(2*np.pi*night_h/24)) if has_night else 0.,
        'night_start_cos': float(np.cos(2*np.pi*night_h/24)) if has_night else 1.,
    }

    if with_direction:
        # Separer la partie day et la partie night pour avoir directions independantes
        parts = s.split(' and ')
        day_part   = next((p for p in parts if 'day'   in p), '')
        night_part = next((p for p in parts if 'night' in p), '')
        result['day_direction']   = int('setup'   in day_part)   - int('setback' in day_part)
        result['night_direction'] = int('setup'   in night_part) - int('setback' in night_part)

    return result


# Heating : pas de direction (toujours setback implicite)
# Cooling  : direction separee par composante Day/Night
PERIOD_COLS = {
    'in.heating_setpoint_offset_period': ('heat', False),
    'in.cooling_setpoint_offset_period': ('cool', True),
}

for col, (prefix, with_dir) in PERIOD_COLS.items():
    if col not in df_enc.columns:
        continue

    encoded = df_enc[col].apply(
        lambda v, wd=with_dir: pd.Series(encode_offset_period_v2(v, wd))
    )

    if with_dir:
        names = [f'{prefix}_period_type',
                 f'{prefix}_day_start_sin',  f'{prefix}_day_start_cos',
                 f'{prefix}_night_start_sin', f'{prefix}_night_start_cos',
                 f'{prefix}_day_direction',   f'{prefix}_night_direction']
    else:
        names = [f'{prefix}_period_type',
                 f'{prefix}_day_start_sin',  f'{prefix}_day_start_cos',
                 f'{prefix}_night_start_sin', f'{prefix}_night_start_cos']

    encoded.columns = names
    df_enc = pd.concat([df_enc.drop(columns=[col]), encoded.astype('float32')], axis=1)
    print(f'{col} -> {names}')

print('Shape :', df_enc.shape)


in.heating_setpoint_offset_period -> ['heat_period_type', 'heat_day_start_sin', 'heat_day_start_cos', 'heat_night_start_sin', 'heat_night_start_cos']


in.cooling_setpoint_offset_period -> ['cool_period_type', 'cool_day_start_sin', 'cool_day_start_cos', 'cool_night_start_sin', 'cool_night_start_cos', 'cool_day_direction', 'cool_night_direction']
Shape : (549971, 382)


## 7. Groupe 4 — HVAC (suite)

| Colonne | Stratégie |
|---|---|
| `in.hvac_heating_type` / `in.hvac_cooling_type` | OHE (section précédente) |
| `in.hvac_heating_efficiency` / `in.hvac_cooling_efficiency` | déjà float (`transformations_numeriques`) |
| `in.hvac_has_ducts` / `in.hvac_has_zonal_electric_heating` / `in.heating_setpoint_has_offset` / `in.cooling_setpoint_has_offset` | déjà Int8 binaire (`transformations_numeriques`) |
| `in.hvac_heating_autosizing_factor` / `in.hvac_cooling_autosizing_factor` / `in.hvac_system_is_faulted` / `in.hvac_system_is_scaled` / `in.hvac_system_single_speed_*` / `in.mechanical_ventilation` / `in.dehumidifier` / `in.natural_ventilation` | supprimés (colonnes constantes) |
| `in.hvac_cooling_partial_space_conditioning` | % float (0.0 – 1.0) |
| `in.hvac_secondary_heating_partial_space_conditioning` | % float (0.0 – 1.0) |
| `in.hvac_secondary_heating_efficiency` | AFUE float |
| `in.heating_fuel` / `in.hvac_has_shared_system` / `in.hvac_secondary_heating_type` / `in.hvac_secondary_heating_fuel` / `in.hvac_shared_efficiencies` | OHE drop_first |

In [12]:
import re

# ── Colonnes constantes → suppression ────────────────────────────────────────
DROP_HVAC_CONST = [
    'in.hvac_heating_autosizing_factor',
    'in.hvac_cooling_autosizing_factor',
    'in.hvac_system_is_faulted',
    'in.hvac_system_is_scaled',
    'in.hvac_system_single_speed_ac_airflow',
    'in.hvac_system_single_speed_ac_charge',
    'in.hvac_system_single_speed_ashp_airflow',
    'in.hvac_system_single_speed_ashp_charge',
    'in.mechanical_ventilation',
    'in.dehumidifier',
    'in.natural_ventilation',
]
dropped = [c for c in DROP_HVAC_CONST if c in df_enc.columns]
df_enc.drop(columns=dropped, inplace=True)
print(f'Supprimés ({len(dropped)}) : {dropped}')

# ── Partial space conditioning → % float ─────────────────────────────────────
def parse_pct_conditioned(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s.lower() == 'none': return 0.0
    m = re.search(r'(\d+)', s)
    return float(m.group(1)) / 100 if m else np.nan

for col in ['in.hvac_cooling_partial_space_conditioning',
            'in.hvac_secondary_heating_partial_space_conditioning']:
    if col in df_enc.columns:
        df_enc[col] = df_enc[col].apply(parse_pct_conditioned).astype('float32')
        print(f'{col} → % float OK  {sorted(df_enc[col].dropna().unique())}')

# ── Secondary heating efficiency → AFUE float ────────────────────────────────
def parse_afue(val):
    if pd.isna(val) or str(val).strip().lower() == 'none': return np.nan
    m = re.search(r'(\d+\.?\d*)%', str(val))
    return float(m.group(1)) / 100 if m else np.nan

col = 'in.hvac_secondary_heating_efficiency'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].apply(parse_afue).astype('float32')
    print(f'secondary_heating_efficiency → AFUE float OK  {sorted(df_enc[col].dropna().unique())}')

# ── OHE ───────────────────────────────────────────────────────────────────────
OHE_HVAC = {
    'in.heating_fuel':                   'heat_fuel',
    'in.hvac_has_shared_system':         'hvac_shared',
    'in.hvac_secondary_heating_type':    'sec_heat_type',
    'in.hvac_secondary_heating_fuel':    'sec_heat_fuel',
    'in.hvac_shared_efficiencies':       'hvac_shared_eff',
}
ohe_present = {k: v for k, v in OHE_HVAC.items() if k in df_enc.columns}
if ohe_present:
    for c in ohe_present:
        df_enc[c] = df_enc[c].astype('category')
    df_enc = pd.get_dummies(df_enc,
                            columns=list(ohe_present.keys()),
                            prefix=list(ohe_present.values()),
                            dtype=np.int8, drop_first=True)
    print(f'OHE HVAC : {list(ohe_present.keys())}')

print(f'\nShape : {df_enc.shape}')

Supprimés (2) : ['in.hvac_system_is_faulted', 'in.hvac_system_is_scaled']


in.hvac_cooling_partial_space_conditioning → % float OK  [np.float32(0.0), np.float32(0.1), np.float32(0.2), np.float32(0.4), np.float32(0.6), np.float32(0.8), np.float32(1.0)]


in.hvac_secondary_heating_partial_space_conditioning → % float OK  [np.float32(0.0), np.float32(0.1), np.float32(0.2), np.float32(0.3), np.float32(0.4), np.float32(0.49)]
secondary_heating_efficiency → AFUE float OK  [np.float32(0.76), np.float32(0.8), np.float32(0.9), np.float32(0.925)]
OHE HVAC : ['in.heating_fuel', 'in.hvac_has_shared_system', 'in.hvac_secondary_heating_type', 'in.hvac_secondary_heating_fuel', 'in.hvac_shared_efficiencies']

Shape : (549971, 394)


## 8. Groupe 5 — Occupants et socio-économique

| Colonne | Stratégie |
|---|---|
| `in.bedrooms` / `in.occupants` / `in.income` / `in.vintage` | déjà numériques (`transformations_numeriques`) |
| `in.units_represented` | supprimé (constante : `'1'`) |
| `in.representative_income` | supprimé (poids d'échantillonnage, pas une feature) |
| `in.tenure` | binaire (Owner=1, Renter=0, Not Available=NaN) |
| `in.vacancy_status` | binaire (Occupied=1, Vacant=0) |
| `in.household_has_tribal_persons` | binaire (Yes=1, No=0, Not Available=NaN) |
| `in.usage_level` | ordinal (Low=0, Medium=1, High=2) |
| `in.federal_poverty_level` | midpoint % → float (ex: `'100-150%'` → 1.25) |
| `in.area_median_income` | midpoint % → float |
| `in.state_metro_median_income` | midpoint % → float |

In [13]:
import re

# ── Constantes + poids d'échantillonnage → suppression ───────────────────────
DROP_SOCIO = ['in.units_represented', 'in.representative_income']
dropped = [c for c in DROP_SOCIO if c in df_enc.columns]
df_enc.drop(columns=dropped, inplace=True)
print(f'Supprimés : {dropped}')

# ── Binaires ─────────────────────────────────────────────────────────────────
BINARY_SOCIO = {
    'in.tenure':                       {'Owner': 1, 'Renter': 0},
    'in.vacancy_status':               {'Occupied': 1, 'Vacant': 0},
    'in.household_has_tribal_persons': {'Yes': 1, 'No': 0},
}
for col, mapping in BINARY_SOCIO.items():
    if col in df_enc.columns:
        df_enc[col] = df_enc[col].map(mapping).astype('Int8')
        print(f'{col.replace("in.", "")} → binaire OK  {df_enc[col].value_counts().to_dict()}')

# ── Usage level → ordinal ─────────────────────────────────────────────────────
col = 'in.usage_level'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map({'Low': 0, 'Medium': 1, 'High': 2}).astype('Int8')
    print(f'usage_level → ordinal OK  {df_enc[col].value_counts().sort_index().to_dict()}')

# ── Revenus relatifs → midpoint % float ──────────────────────────────────────
# '0-100%' → 0.50  |  '100-150%' → 1.25  |  '400%+' → 4.0  |  'Not Available' → NaN
def parse_pct_range(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s == 'Not Available': return np.nan
    if s.endswith('%+'):
        return float(s[:-2]) / 100
    m = re.match(r'(\d+)-(\d+)%', s)
    if m: return (float(m.group(1)) + float(m.group(2))) / 2 / 100
    m2 = re.match(r'(\d+)%', s)
    if m2: return float(m2.group(1)) / 100
    return np.nan

INCOME_PCT_COLS = [
    'in.federal_poverty_level',
    'in.area_median_income',
    'in.state_metro_median_income',
]
for col in INCOME_PCT_COLS:
    if col in df_enc.columns:
        df_enc[col] = df_enc[col].apply(parse_pct_range).astype('float32')
        print(f'{col.replace("in.", "")} → % float OK  {sorted(df_enc[col].dropna().unique())}')

print(f'\nShape : {df_enc.shape}')

Supprimés : ['in.units_represented', 'in.representative_income']
tenure → binaire OK  {np.int8(1): 307512, np.int8(0): 175822}
vacancy_status → binaire OK  {np.int8(1): 483334, np.int8(0): 66637}
household_has_tribal_persons → binaire OK  {np.int8(0): 480395, np.int8(1): 2939}
usage_level → ordinal OK  {np.int8(0): 137495, np.int8(1): 274985, np.int8(2): 137491}


federal_poverty_level → % float OK  [np.float32(0.5), np.float32(1.25), np.float32(1.75), np.float32(2.5), np.float32(3.5), np.float32(4.0)]


area_median_income → % float OK  [np.float32(0.15), np.float32(0.45), np.float32(0.7), np.float32(0.9), np.float32(1.1), np.float32(1.35), np.float32(1.5)]


state_metro_median_income → % float OK  [np.float32(0.15), np.float32(0.45), np.float32(0.7), np.float32(0.9), np.float32(1.1), np.float32(1.35), np.float32(1.5)]

Shape : (549971, 392)


## 9. Groupe 6 — Véhicule électrique

| Colonne | Stratégie |
|---|---|
| `in.electric_vehicle_ownership` / `in.electric_vehicle_outlet_access` | binaire (Yes=1, No=0) |
| `in.electric_vehicle_charger` | ordinal (None=0, Level 1=1, Level 2=2) |
| `in.electric_vehicle_charge_at_home` | midpoint % → float |
| `in.electric_vehicle_miles_traveled` | numérique direct |
| `in.electric_vehicle_battery` | décomposé → `ev_vehicle_type` (OHE) + `ev_battery_range_miles` (numérique) |

In [14]:
import re

# ── Binaires ──────────────────────────────────────────────────────────────────
for col in ['in.electric_vehicle_ownership', 'in.electric_vehicle_outlet_access']:
    if col in df_enc.columns:
        df_enc[col] = df_enc[col].map({'Yes': 1, 'No': 0}).astype('Int8')
        print(f'{col.replace("in.", "")} → binaire OK  {df_enc[col].value_counts().to_dict()}')

# ── Charger → ordinal (puissance croissante) ──────────────────────────────────
CHARGER_MAP = {'None': 0, 'Level 1 charger': 1, 'Level 2 charger': 2}
col = 'in.electric_vehicle_charger'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map(CHARGER_MAP).astype('Int8')
    print(f'electric_vehicle_charger → ordinal OK  {df_enc[col].value_counts().sort_index().to_dict()}')

# ── Charge at home → midpoint % float ────────────────────────────────────────
def parse_charge_pct(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s == '100%': return 1.0
    m = re.match(r'(\d+)-(\d+)%', s)
    if m: return (float(m.group(1)) + float(m.group(2))) / 2 / 100
    return np.nan

col = 'in.electric_vehicle_charge_at_home'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].apply(parse_charge_pct).astype('float32')
    print(f'in.electric_vehicle_charge_at_home -> % float OK  {sorted(df_enc[col].dropna().unique())}')

# ── Miles traveled → numérique annuel ────────────────────────────────────────
# Valeurs : 1000 – 22500 miles/an (moyenne US ≈ 13 500 miles/an)
col = 'in.electric_vehicle_miles_traveled'
if col in df_enc.columns:
    df_enc[col] = pd.to_numeric(df_enc[col], errors='coerce').astype('float32')
    print(f'in.electric_vehicle_miles_traveled -> miles/an OK  {sorted(df_enc[col].dropna().unique())}')

# ── Battery → supprimé ───────────────────────────────────────────────────────
if 'in.electric_vehicle_battery' in df_enc.columns:
    df_enc.drop(columns=['in.electric_vehicle_battery'], inplace=True)
    print('in.electric_vehicle_battery supprimé')

print(f'\nShape : {df_enc.shape}')

electric_vehicle_ownership → binaire OK  {}
electric_vehicle_outlet_access → binaire OK  {}
electric_vehicle_charger → ordinal OK  {np.int8(0): 544193, np.int8(1): 3069, np.int8(2): 2709}


in.electric_vehicle_charge_at_home -> % float OK  [np.float32(0.095), np.float32(0.295), np.float32(0.495), np.float32(0.695), np.float32(0.895), np.float32(1.0)]
in.electric_vehicle_miles_traveled -> miles/an OK  [np.float32(1000.0), np.float32(3000.0), np.float32(5000.0), np.float32(7000.0), np.float32(9000.0), np.float32(11000.0), np.float32(13000.0), np.float32(15000.0), np.float32(17000.0), np.float32(19000.0), np.float32(22500.0)]
in.electric_vehicle_battery supprimé

Shape : (549971, 391)


## 10. Groupe 7 — Solaire et stockage

| Colonne | Stratégie |
|---|---|
| `in.battery` | supprimé (100% "None") |
| `in.has_pv` | binaire (Yes=1, No=0) |
| `in.pv_system_size` | numérique kWDC (`'None'` → 0) |
| `in.pv_orientation` | sin/cos circulaire (`'None'` → NaN) — 99% None, pertinent uniquement pour les 1% avec panneaux |

In [15]:
import re

# Battery -> suppression (100% "None") 
if 'in.battery' in df_enc.columns:
    df_enc.drop(columns=['in.battery'], inplace=True)
    print('battery supprimé (constante)')

# has_pv -> binaire 
col = 'in.has_pv'
if col in df_enc.columns:
    # Deja converti en Int8 (0/1) par transformations_numeriques
    if not pd.api.types.is_integer_dtype(df_enc[col]):
        df_enc[col] = df_enc[col].map({'Yes': 1, 'No': 0}).astype('Int8')
    n_pv = df_enc[col].sum()
    print(f'has_pv -> binaire OK  ({n_pv} batiments avec PV, {n_pv/len(df_enc)*100:.1f}%)')

# PV system size → kWDC numérique ('None' → 0) 
col = 'in.pv_system_size'
if col in df_enc.columns:
    def parse_pv_size(val):
        if pd.isna(val) or str(val).strip().lower() == 'none': return 0.0
        m = re.search(r'(\d+\.?\d*)', str(val))
        return float(m.group(1)) if m else 0.0
    df_enc[col] = df_enc[col].apply(parse_pv_size).astype('float32')
    print(f'pv_system_size -> kWDC OK  {sorted(df_enc[col].unique())}')

#  PV orientation -> sin/cos ('None' → NaN) 
ORIENT_DEG = {
    'North': 0, 'Northeast': 45, 'East': 90, 'Southeast': 135,
    'South': 180, 'Southwest': 225, 'West': 270, 'Northwest': 315,
}
col = 'in.pv_orientation'
if col in df_enc.columns:
    deg = df_enc[col].map(ORIENT_DEG)   # None -> NaN
    df_enc['in.pv_orientation_sin'] = np.sin(np.radians(deg)).astype('float32')
    df_enc['in.pv_orientation_cos'] = np.cos(np.radians(deg)).astype('float32')
    df_enc.drop(columns=[col], inplace=True)
    print('in.pv_orientation -> sin/cos OK  (NaN pour les 99% sans panneaux)')

print(f'\nShape : {df_enc.shape}')

has_pv -> binaire OK  (5443 batiments avec PV, 1.0%)


pv_system_size -> kWDC OK  [np.float32(0.0), np.float32(1.0), np.float32(3.0), np.float32(5.0), np.float32(7.0), np.float32(9.0), np.float32(11.0), np.float32(13.0)]
in.pv_orientation -> sin/cos OK  (NaN pour les 99% sans panneaux)

Shape : (549971, 392)


## 11. Groupe 8: Eau Chaude
| Colonne                           | Stratégie                                                                                                                                                                                                                                                                                                           |
| --------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `in.water_heater_efficiency`      | numérique (rendement/UEF) : mapping des valeurs connues (`Electric Premium` → 0.95, `Natural Gas Standard` → 0.59, etc.), extraction automatique du UEF pour les chauffe-eau à pompe à chaleur (`3.45 UEF` → 3.45), valeurs par défaut pour les modèles tankless et solaires thermiques ; valeurs inconnues → `NaN` |
| `in.water_heater_technology`      | catégorielle : regroupement en `heat_pump`, `tankless`, `solar`, `indirect`, `storage` (par défaut) ; ensuite One-Hot Encoding (`drop_first=True`)                                                                                                                                                                  |
| `in.solar_collector_area`         | numérique (surface en sqft) : extraction de la surface du collecteur solaire à partir de la chaîne (`40 sqft` → 40) + la conversion en m2 ; absence d'information → 0                                                                                                                                                                     |
| `in.water_heater_orientation_deg` | angle d'orientation : `North=0°`, `East=90°`, `South=180°`, `West=270°` ; orientation non détectée → `NaN`                                                                                                                                                                                                          |
| `in.water_heater_orientation_sin` | encodage cyclique : sinus de l'orientation en radians ; préserve la continuité angulaire                                                                                                                                                                                                                            |
| `in.water_heater_orientation_cos` | encodage cyclique : cosinus de l'orientation en radians ; complément du sinus pour représenter l'orientation                                                                                                                                                                                                        |
| `in.water_heater_fuel`            | catégorielle : harmonisation des combustibles (`Electricity` → `electricity`, `Natural Gas` → `natural_gas`, etc.) ; ensuite One-Hot Encoding (`drop_first=True`)                                                                                                                                                   |
| `in.water_heater_in_unit`         | binaire (`Yes=1`, `No=0`) indiquant si le chauffe-eau est situé dans le logement                                                                                                                                                                                                                                    |
| `in.water_heater_location`        | catégorielle : regroupement thermique en `conditioned`, `semi_conditioned`, `unconditioned`, `unknown` selon l'emplacement ; ensuite One-Hot Encoding (`drop_first=True`)                                                                                                                                           |
| `in.water_heater_technology_*`    | variables indicatrices créées par One-Hot Encoding des technologies (`heat_pump`, `tankless`, `solar`, etc.)                                                                                                                                                                                                        |
| `in.water_heater_fuel_*`          | variables indicatrices créées par One-Hot Encoding des combustibles (`natural_gas`, `propane`, `fuel_oil`, etc.)                                                                                                                                                                                                    |
| `in.water_heater_location_*`      | variables indicatrices créées par One-Hot Encoding des groupes de localisation thermique (`semi_conditioned`, `unconditioned`, `unknown`, etc.)                                                                                                                                                                     |


In [16]:

water_heater_raw = df_enc["in.water_heater_efficiency"].copy()
efficiency_map = {
    "Electric Premium": 0.95,
    "Electric Standard": 0.92,
    "Fuel Oil Premium": 0.68,
    "Fuel Oil Standard": 0.62,
    "Natural Gas Premium": 0.67,
    "Natural Gas Standard": 0.59,
    "Propane Premium": 0.67,
    "Propane Standard": 0.59,
    "Electric Heat Pump, 50 gal, 3.45 UEF": 3.45,
}

def water_heater_efficiency(eff_str):
    if pd.isna(eff_str):
        return np.nan
    eff_str = str(eff_str)
    # Heat Pump
    m = re.search(r'([\d.]+)\s*UEF', eff_str)
    if m:
        return float(m.group(1))
    # Valeurs connues
    if eff_str in efficiency_map:
        return efficiency_map[eff_str]
    # Valeurs typiques tankless
    if "Electric Tankless" in eff_str:
        return 0.99
    if "Natural Gas Tankless" in eff_str:
        return 0.82   # Table 137 guide ResStock
    if "Propane Tankless" in eff_str:
        return 0.82   # Table 137 guide ResStock
    if "FIXME Fuel Oil Indirect" in eff_str:
        return 0.62   # Table 137 : Fuel Oil storage
    if "Other Fuel" in eff_str:
        return 0.59   # Table 137 : coal storage
    if "Wood" in eff_str:
        return 0.59   # Table 137 : wood storage
    # Solar thermique
    if "Solar Thermal" in eff_str:
        return 4.0    # valeur proxy (solar fraction > 1 par convention)
    return np.nan

df_enc["in.water_heater_efficiency"] = water_heater_raw.apply(water_heater_efficiency)

# ============================================================
# 2. Technologie du chauffe-eau
# ============================================================

def water_heater_technology(eff_str):
    if pd.isna(eff_str):
        return "unknown"
    eff_str = str(eff_str)
    if "Heat Pump" in eff_str:
        return "heat_pump"
    if "Tankless" in eff_str:
        return "tankless"
    if "Solar Thermal" in eff_str:
        return "solar"
    if "Indirect" in eff_str:
        return "indirect"
    return "storage"

df_enc["in.water_heater_technology"] = water_heater_raw.apply(water_heater_technology)

# ============================================================
# 3. Surface collecteur solaire
# ============================================================

def water_heater_collector_area(eff_str):
    if pd.isna(eff_str):
        return 0.0

    eff_str = str(eff_str)

    m = re.search(r'(\d+)\s*sqft', eff_str)
    if m:
        area_sqft = float(m.group(1))
        area_m2 = area_sqft * 0.092903
        return area_m2

    return 0.0
df_enc["in.solar_collector_area"] = water_heater_raw.apply(water_heater_collector_area)

# ============================================================
# 4. Orientation solaire
# ============================================================

orientation_map = {
    "North": 0,
    "East": 90,
    "South": 180,
    "West": 270
}

def water_heater_orientation(or_str):
    if pd.isna(or_str):
        return np.nan
    or_str = str(or_str)
    for direction, angle in orientation_map.items():
        if direction in or_str:
            return angle
    return np.nan

df_enc["in.water_heater_orientation_deg"] = water_heater_raw.apply(water_heater_orientation)


# Encodage cyclique
radians = np.deg2rad(df_enc["in.water_heater_orientation_deg"])

df_enc["in.water_heater_orientation_sin"] = np.sin(radians)
df_enc["in.water_heater_orientation_cos"] = np.cos(radians)

# ============================================================
# 5. Combustible
# ============================================================

fuel_map = {
    "Electricity": "electricity",
    "Natural Gas": "natural_gas",
    "Propane": "propane",
    "Fuel Oil": "fuel_oil",
    "Solar Thermal": "solar",
    "Wood": "wood",
    "Other Fuel": "other"
}

df_enc["in.water_heater_fuel"] = df_enc["in.water_heater_fuel"].map(fuel_map)

# ============================================================
# 6. Présence dans le logement
# ============================================================

# Deja converti en Int8 (0/1) par transformations_numeriques
# map({"Yes":1,"No":0}) retournerait NaN -> on garde la colonne telle quelle
if df_enc["in.water_heater_in_unit"].dtype.name not in ("Int8", "int8", "int64"):
    df_enc["in.water_heater_in_unit"] = df_enc["in.water_heater_in_unit"].map({"Yes": 1, "No": 0})


# ============================================================
# 7. Localisation thermique
# ============================================================

conditioned = {
    "Living Space",
    "Conditioned Mechanical Room",
    "Heated Basement"
}

semi_conditioned = {
    "Garage",
    "Attic"
}

unconditioned = {
    "Outside",
    "Crawlspace",
    "Unheated Basement"
}

def location_group(loc_str):
    if pd.isna(loc_str):
        return "unknown"
    loc_str = str(loc_str)
    if loc_str in conditioned:
        return "conditioned"
    if loc_str in semi_conditioned:
        return "semi_conditioned"
    if loc_str  in unconditioned:
        return "unconditioned"
    return "unknown"

df_enc["in.water_heater_location"] =df_enc["in.water_heater_location"].apply(location_group)


# ============================================================
# 8. One-Hot Encoding
# ============================================================

categorical_cols = [
    "in.water_heater_technology",
    "in.water_heater_fuel",
    "in.water_heater_location"
]
for col in categorical_cols:
    print(col, col in df_enc.columns)

df_enc = pd.get_dummies(
    df_enc,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)


print("Features chauffe-eau créées avec succès.")

modified_columns = [
    "in.water_heater_efficiency",
    "in.water_heater_fuel",
    "in.water_heater_in_unit",
    "in.water_heater_location"
]
print("\nColonnes modifiées :")

for col in modified_columns:
    if col in df_enc.columns:
        print(f"  * {col}")

print(f"\nShape finale : {df_enc.shape}")

in.water_heater_technology True
in.water_heater_fuel True
in.water_heater_location True
Features chauffe-eau créées avec succès.

Colonnes modifiées :
  * in.water_heater_efficiency
  * in.water_heater_in_unit

Shape finale : (549971, 406)


## 12. Groupe 9 — Panneau électrique

| Colonne | Stratégie |
|---|---|
| `in.electric_panel_service_rating..a` | déjà float — ampérage du panneau (60–400A) → float32 |
| `in.electric_panel_breaker_space_total_count` | déjà float — nombre d'emplacements disjoncteurs (8–60) → float32 |

In [17]:
for col in ['in.electric_panel_service_rating..a',
            'in.electric_panel_breaker_space_total_count']:
    if col in df_enc.columns:
        df_enc[col] = df_enc[col].astype('float32')
        print(f'{col.replace("in.", "")} → float32 OK  {sorted(df_enc[col].dropna().unique())}')

print(f'\nShape : {df_enc.shape}')

electric_panel_service_rating..a → float32 OK  [np.float32(60.0), np.float32(90.0), np.float32(100.0), np.float32(120.0), np.float32(125.0), np.float32(150.0), np.float32(200.0), np.float32(250.0), np.float32(300.0), np.float32(400.0)]
electric_panel_breaker_space_total_count → float32 OK  [np.float32(8.0), np.float32(9.0), np.float32(10.0), np.float32(11.0), np.float32(12.0), np.float32(13.0), np.float32(14.0), np.float32(15.0), np.float32(16.0), np.float32(17.0), np.float32(18.0), np.float32(19.0), np.float32(20.0), np.float32(21.0), np.float32(22.0), np.float32(23.0), np.float32(24.0), np.float32(25.0), np.float32(26.0), np.float32(27.0), np.float32(28.0), np.float32(29.0), np.float32(30.0), np.float32(31.0), np.float32(32.0), np.float32(33.0), np.float32(34.0), np.float32(35.0), np.float32(36.0), np.float32(37.0), np.float32(38.0), np.float32(39.0), np.float32(40.0), np.float32(41.0), np.float32(42.0), np.float32(43.0), np.float32(44.0), np.float32(45.0), np.float32(46.0), np.float

## 13. Groupe 10 — Appareils

| Colonne originale            | Remplacée par                                                                                              |
| ---------------------------- | ---------------------------------------------------------------------------------------------------------- |
| `in.clothes_dryer`           | `in.clothes_dryer_efficiency`, `in.clothes_dryer_has`, `in.clothes_dryer_electric`, `in.clothes_dryer_gas` |
| `in.clothes_washer`          | `in.clothes_washer_efficiency`                                                                             |
| `in.clothes_washer_presence` | `in.clothes_washer_has`                                                                                    |
| `in.cooking_range`           | variables One-Hot `cooking_type_*`                                                                         |
| `in.refrigerator`            | `in.refrigerator_ef`, `in.refrigerator_has`                                                                |
| `in.lighting`                | `in.lighting_efficiency`                                                                                   |
| `in.misc_extra_refrigerator` | `in.misc_extra_refrigerator_ef`, `in.misc_extra_refrigerator_has`                                          |
| `in.misc_freezer`            | `in.misc_freezer_ef`                                                                                       |
| `in.misc_gas_fireplace`      | `in.misc_gas_fireplace_present`                                                                            |
| `in.misc_gas_grill`          | `in.misc_gas_grill_present`                                                                                |
| `in.misc_gas_lighting`       | `in.misc_gas_lighting_present`                                                                             |
| `in.misc_hot_tub_spa`        | `in.has_hot_tub`, `in.hot_tub_electric`, `in.misc_hot_tub_gas`                                             |
| `in.misc_pool`               | `in.has_pool`                                                                                              |
| `in.misc_pool_heater`        | `in.pool_heat_pump`, `in.pool_heater_electric`, `in.pool_heater_gas`                                       |
| `in.misc_pool_pump`          | `in.misc_pool_pump_hp`                                                                                     |
| `in.misc_well_pump`          | `in.has_well_pump`                                                                                         |
| `in.ceiling_fan`             | `in.has_ceiling_fan`, `in.ceiling_fan_used`                                                                |

In [18]:
import re

# ── Clothes dryer ─────────────────────────────────────────────────────────────
dryer_eff = {
    "Electric": 2.7,
    "Gas": 2.39,
    "Propane": 2.39,
    "None": 0
}
if 'in.clothes_dryer' in df_enc.columns:
    df_enc["in.clothes_dryer_efficiency"] = df_enc["in.clothes_dryer"].map(dryer_eff).fillna(0)
    df_enc["in.clothes_dryer_has"]        = (df_enc["in.clothes_dryer"] != "None").astype(int)
    df_enc["in.clothes_dryer_electric"]   = (df_enc["in.clothes_dryer"] == "Electric").astype(int)
    df_enc["in.clothes_dryer_gas"]        = (df_enc["in.clothes_dryer"].isin(["Gas", "Propane"])).astype(int)
    print("clothes_dryer → 4 colonnes OK")

# ── Clothes washer ────────────────────────────────────────────────────────────
washer_eff = {
    "EnergyStar": 2.07,
    "Standard": 0.95,
    "None": 0
}
if 'in.clothes_washer' in df_enc.columns:
    df_enc["in.clothes_washer_efficiency"] = df_enc["in.clothes_washer"].map(washer_eff).fillna(0)
    print("clothes_washer → efficiency OK")

if 'in.clothes_washer_presence' in df_enc.columns:
    df_enc["in.clothes_washer_has"] = (df_enc["in.clothes_washer_presence"] == "Yes").astype(int)
    print("clothes_washer_presence → has OK")

# ── Cooking range → OHE ───────────────────────────────────────────────────────
fuel_map = {
    "Electric Induction":  "induction",
    "Electric Resistance": "electric",
    "Gas":                 "gas",
    "Propane":             "propane",
    "None":                "none"
}
if 'in.cooking_range' in df_enc.columns:
    df_enc["cooking_type"] = df_enc["in.cooking_range"].map(fuel_map)
    df_enc = pd.get_dummies(df_enc, columns=["cooking_type"], drop_first=True, dtype=np.int8)
    print("cooking_range → cooking_type_* OHE OK")

# ── Refrigerator ──────────────────────────────────────────────────────────────
def refrigerator_efficiency(ref_str):
    if pd.isna(ref_str) or ref_str == "None":
        return 0
    m = re.search(r'([\d.]+)', str(ref_str))
    return float(m.group(1)) if m else 0

if 'in.refrigerator' in df_enc.columns:
    df_enc["in.refrigerator_ef"]  = df_enc["in.refrigerator"].apply(refrigerator_efficiency)
    df_enc["in.refrigerator_has"] = (df_enc["in.refrigerator"] != "None").astype(int)
    print("refrigerator → ef + has OK")

# ── Lighting ──────────────────────────────────────────────────────────────────
lighting_score = {
    "100% Incandescent": 1,
    "100% CFL":          2,
    "100% LED":          3
}
if 'in.lighting' in df_enc.columns:
    df_enc["in.lighting_efficiency"] = df_enc["in.lighting"].map(lighting_score)
    print("lighting → efficiency OK")

# ── Extra refrigerator ────────────────────────────────────────────────────────
if 'in.misc_extra_refrigerator' in df_enc.columns:
    df_enc["in.misc_extra_refrigerator_ef"]  = df_enc["in.misc_extra_refrigerator"].apply(refrigerator_efficiency)
    df_enc["in.misc_extra_refrigerator_has"] = (df_enc["in.misc_extra_refrigerator"] != "None").astype(int)
    print("misc_extra_refrigerator → ef + has OK")

# ── Freezer ───────────────────────────────────────────────────────────────────
if 'in.misc_freezer' in df_enc.columns:
    df_enc["in.misc_freezer_ef"] = df_enc["in.misc_freezer"].apply(refrigerator_efficiency)
    print("misc_freezer → ef OK")

# ── Binary gas appliances ─────────────────────────────────────────────────────
for col in ["in.misc_gas_fireplace", "in.misc_gas_grill", "in.misc_gas_lighting"]:
    if col in df_enc.columns:
        df_enc[col + "_present"] = (df_enc[col] != "None").astype(int)
print("misc_gas_* → present OK")

# ── Hot tub / spa ─────────────────────────────────────────────────────────────
if 'in.misc_hot_tub_spa' in df_enc.columns:
    df_enc["in.has_hot_tub"]       = (df_enc["in.misc_hot_tub_spa"] != "None").astype(int)
    df_enc["in.hot_tub_electric"]  = (df_enc["in.misc_hot_tub_spa"] == "Electricity").astype(int)
    df_enc["in.misc_hot_tub_gas"]  = (df_enc["in.misc_hot_tub_spa"] == "Natural Gas").astype(int)
    print("misc_hot_tub_spa → 3 colonnes OK")

# ── Pool ──────────────────────────────────────────────────────────────────────
if 'in.misc_pool' in df_enc.columns:
    df_enc["in.has_pool"] = (df_enc["in.misc_pool"] == "Has Pool").astype(int)
    print("misc_pool → has_pool OK")

# ── Pool heater ───────────────────────────────────────────────────────────────
if 'in.misc_pool_heater' in df_enc.columns:
    df_enc["in.pool_heater_present"] = (df_enc["in.misc_pool_heater"] != "None").astype(int)
    df_enc["in.pool_heat_pump"]      = (df_enc["in.misc_pool_heater"] == "Electric Heat Pump").astype(int)
    df_enc["in.pool_heater_electric"] = (df_enc["in.misc_pool_heater"] == "Electricity").astype(int)
    df_enc["in.pool_heater_gas"]      = (df_enc["in.misc_pool_heater"] == "Natural Gas").astype(int)
    print("misc_pool_heater → 3 colonnes OK")

# ── Pool pump → HP numérique ──────────────────────────────────────────────────
def extract_hp(x):
    if pd.isna(x) or x == "None":
        return 0
    m = re.search(r'([\d.]+)', str(x))
    return float(m.group(1)) if m else 0

if 'in.misc_pool_pump' in df_enc.columns:
    df_enc["in.misc_pool_pump_hp"] = df_enc["in.misc_pool_pump"].apply(extract_hp)
    print("misc_pool_pump → hp OK")

# ── Well pump ─────────────────────────────────────────────────────────────────
if 'in.misc_well_pump' in df_enc.columns:
    df_enc["in.has_well_pump"] = (df_enc["in.misc_well_pump"] != "None").astype(int)
    print("misc_well_pump → has_well_pump OK")

# ── Ceiling fan ───────────────────────────────────────────────────────────────
if 'in.ceiling_fan' in df_enc.columns:
    df_enc["in.has_ceiling_fan"]   = (df_enc["in.ceiling_fan"] != "None").astype(int)
    df_enc["in.ceiling_fan_used"]  = (df_enc["in.ceiling_fan"] == "Standard Efficiency").astype(int)
    print("ceiling_fan → has + used OK")

# ── Suppression colonnes originales ───────────────────────────────────────────
cols_to_drop = [
    "in.clothes_dryer",
    "in.clothes_washer",
    "in.clothes_washer_presence",
    "in.cooking_range",
    "in.refrigerator",
    "in.lighting",
    "in.misc_extra_refrigerator",
    "in.misc_freezer",
    "in.misc_gas_fireplace",
    "in.misc_gas_grill",
    "in.misc_gas_lighting",
    "in.misc_hot_tub_spa",
    "in.misc_pool",
    "in.misc_pool_heater",
    "in.misc_pool_pump",
    "in.misc_well_pump",
    "in.ceiling_fan",
]
dropped = [c for c in cols_to_drop if c in df_enc.columns]
df_enc.drop(columns=dropped, inplace=True)
print(f'\nSupprimés ({len(dropped)}) : {dropped}')
print(f'Shape : {df_enc.shape}')

clothes_dryer → 4 colonnes OK
clothes_washer → efficiency OK
clothes_washer_presence → has OK


cooking_range → cooking_type_* OHE OK


C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\859819319.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enc["in.clothes_dryer_efficiency"] = df_enc["in.clothes_dryer"].map(dryer_eff).fillna(0)
C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\859819319.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enc["in.clothes_dryer_has"]        = (df_enc["in.clothes_dryer"] != "None").astype(int)
C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\859819319.py:13: PerformanceWarning: DataFram

C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\859819319.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enc["in.refrigerator_ef"]  = df_enc["in.refrigerator"].apply(refrigerator_efficiency)
C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\859819319.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enc["in.refrigerator_has"] = (df_enc["in.refrigerator"] != "None").astype(int)
C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\859819319.py:63: PerformanceWarning: DataFrame is highly 

refrigerator → ef + has OK
lighting → efficiency OK


...

misc_extra_refrigerator → ef + has OK


C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\859819319.py:74: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enc["in.misc_freezer_ef"] = df_enc["in.misc_freezer"].apply(refrigerator_efficiency)
C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\859819319.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enc[col + "_present"] = (df_enc[col] != "None").astype(int)
C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\859819319.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is

misc_freezer → ef OK
misc_gas_* → present OK
misc_hot_tub_spa → 3 colonnes OK
misc_pool → has_pool OK
misc_pool_heater → 3 colonnes OK


misc_pool_pump → hp OK
misc_well_pump → has_well_pump OK
ceiling_fan → has + used OK

Supprimés (17) : ['in.clothes_dryer', 'in.clothes_washer', 'in.clothes_washer_presence', 'in.cooking_range', 'in.refrigerator', 'in.lighting', 'in.misc_extra_refrigerator', 'in.misc_freezer', 'in.misc_gas_fireplace', 'in.misc_gas_grill', 'in.misc_gas_lighting', 'in.misc_hot_tub_spa', 'in.misc_pool', 'in.misc_pool_heater', 'in.misc_pool_pump', 'in.misc_well_pump', 'in.ceiling_fan']
Shape : (549971, 420)


C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\859819319.py:111: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enc["in.misc_pool_pump_hp"] = df_enc["in.misc_pool_pump"].apply(extract_hp)
C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\859819319.py:116: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enc["in.has_well_pump"] = (df_enc["in.misc_well_pump"] != "None").astype(int)
C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\859819319.py:121: PerformanceWarning: DataFrame is highly fragment

## 14. Conversion : `in.dishwasher`

| Colonne originale | Remplacée par |
| ----------------- | ------------- |
| `in.dishwasher`   | `dishwasher_kwh` (ex: `"290 Rated kWh"` → `290.0`), `has_dishwasher` (binaire) |

In [19]:
def dishwasher_kwh(ds_str):
    if pd.isna(ds_str) or ds_str == "None":
        return 0
    m = re.search(r'(\d+)', str(ds_str))
    return float(m.group(1)) if m else 0

if 'in.dishwasher' in df_enc.columns:
    df_enc["in.dishwasher_kwh"] = df_enc["in.dishwasher"].apply(dishwasher_kwh)
    df_enc["in.has_dishwasher"] = (df_enc["in.dishwasher"] != "None").astype(int)
    df_enc.drop(columns=["in.dishwasher"], inplace=True)
    print(f"dishwasher → kwh + has OK  {sorted(df_enc['in.dishwasher_kwh'].unique())}")

print(f'Shape : {df_enc.shape}')

dishwasher → kwh + has OK  [np.float64(0.0), np.float64(290.0), np.float64(318.0)]
Shape : (549971, 421)


C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\2401404193.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enc["in.dishwasher_kwh"] = df_enc["in.dishwasher"].apply(dishwasher_kwh)
C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\2401404193.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enc["in.has_dishwasher"] = (df_enc["in.dishwasher"] != "None").astype(int)


## Sauvegarde → `features_encodees.parquet`

## 16. Colonnes string résiduelles

In [20]:
import re

# ── census_region → supprimé (redondant avec zone climatique) ─────────────────
if 'in.census_region' in df_enc.columns:
    df_enc.drop(columns=['in.census_region'], inplace=True)
    print('census_region supprimé')

# ── Heures de ventilation → sin/cos circulaire ────────────────────────────────
for col, prefix in [('in.bathroom_spot_vent_hour', 'bath_vent'),
                    ('in.range_spot_vent_hour',    'range_vent')]:
    if col in df_enc.columns:
        hour = pd.to_numeric(df_enc[col], errors='coerce').astype(float)  # deja int64
        df_enc[f'{prefix}_sin'] = np.sin(2 * np.pi * hour / 24).astype('float32')
        df_enc[f'{prefix}_cos'] = np.cos(2 * np.pi * hour / 24).astype('float32')
        df_enc.drop(columns=[col], inplace=True)
        print(f'{col} -> sin/cos OK')

# ── Colonnes "% Usage" → float ────────────────────────────────────────────────
USAGE_COLS = [
    'in.clothes_dryer_usage_level',
    'in.clothes_washer_usage_level',
    'in.cooking_range_usage_level',
    'in.dishwasher_usage_level',
    'in.refrigerator_usage_level',
    # 'in.hot_water_fixtures',  # deja float64 (0.6-2.0) depuis transformations_numeriques
    'in.plug_load_diversity',
    'in.plug_loads',
]
def parse_pct_usage(val):
    if pd.isna(val): return np.nan
    m = re.search(r'(\d+)', str(val))
    return float(m.group(1)) / 100 if m else np.nan

for col in USAGE_COLS:
    if col in df_enc.columns:
        # Deja float64 (converti par transformations_numeriques) -> skip
        if pd.api.types.is_float_dtype(df_enc[col]) or pd.api.types.is_integer_dtype(df_enc[col]):
            print(f'{col.replace("in.", "")} deja numerique, skip')
            continue
        df_enc[col] = df_enc[col].apply(parse_pct_usage).astype('float32')
        print(f'{col.replace("in.", "")} -> % float OK')

# ── Jours d'indisponibilité → numérique ───────────────────────────────────────
DAYS_MAP = {
    'Never': 0, '1 day': 1, '3 days': 3, '1 week': 7,
    '2 weeks': 14, '1 month': 30, '3 months': 90, 'Year round': 365,
}
for col in ['in.cooling_unavailable_days', 'in.heating_unavailable_days']:
    if col in df_enc.columns:
        df_enc[col] = df_enc[col].map(DAYS_MAP).astype('float32')
        print(f'{col.replace("in.", "")} -> jours OK')

# ── Duct leakage → leakage_pct (float) + duct_insulated (binaire) ─────────────
col = 'in.duct_leakage_and_insulation'
if col in df_enc.columns:
    def parse_duct(val):
        if pd.isna(val) or str(val).strip() == 'None':
            return np.nan, np.nan
        m = re.search(r'(\d+)%', str(val))
        leakage = float(m.group(1)) / 100 if m else np.nan
        insulated = int('Insulated' in val and 'Uninsulated' not in val)
        return leakage, insulated
    parsed = df_enc[col].apply(lambda v: pd.Series(parse_duct(v),
                               index=['duct_leakage_pct', 'duct_insulated']))
    df_enc = pd.concat([df_enc.drop(columns=[col]), parsed.astype('float32')], axis=1)
    print('duct_leakage_and_insulation -> duct_leakage_pct + duct_insulated OK')

# ── Duct location → groupes thermiques + OHE ─────────────────────────────────
DUCT_LOC_MAP = {
    'Living Space':                'conditioned',
    'Heated Basement':             'conditioned',
    'Conditioned Mechanical Room': 'conditioned',
    'Attic':                       'semi_conditioned',
    'Garage':                      'semi_conditioned',
    'Crawlspace':                  'unconditioned',
    'Unheated Basement':           'unconditioned',
    'None':                        'none',
}
col = 'in.duct_location'
if col in df_enc.columns:
    df_enc[col] = df_enc[col].map(DUCT_LOC_MAP).astype('category')
    df_enc = pd.get_dummies(df_enc, columns=[col], prefix='duct_loc',
                            dtype=np.int8, drop_first=True)
    print('duct_location -> duct_loc_* OHE OK')

print(f'\nShape : {df_enc.shape}')


census_region supprimé
in.bathroom_spot_vent_hour -> sin/cos OK
in.range_spot_vent_hour -> sin/cos OK
clothes_dryer_usage_level deja numerique, skip
clothes_washer_usage_level deja numerique, skip
cooking_range_usage_level deja numerique, skip
dishwasher_usage_level deja numerique, skip
refrigerator_usage_level deja numerique, skip
plug_load_diversity deja numerique, skip
plug_loads deja numerique, skip
cooling_unavailable_days -> jours OK
heating_unavailable_days -> jours OK

Shape : (549971, 422)


C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\1280639927.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enc[f'{prefix}_sin'] = np.sin(2 * np.pi * hour / 24).astype('float32')
C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\1280639927.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_enc[f'{prefix}_cos'] = np.cos(2 * np.pi * hour / 24).astype('float32')
C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\1280639927.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is

In [21]:
# ── in.duct_location → label encoding (7 catégories nominales) ──────────────────
duct_loc_map = {
    'None':              0,
    'Living Space':      1,
    'Attic':             2,
    'Crawlspace':        3,
    'Heated Basement':   4,
    'Unheated Basement': 5,
    'Garage':            6,
}
if 'in.duct_location' in df_enc.columns:
    df_enc["in.duct_location"] = (
        df_enc["in.duct_location"].map(duct_loc_map).astype("Int8")
    )
    print('in.duct_location encodé (label)')

# ── in.duct_leakage_and_insulation → 2 features numériques ─────────────────
# Format : 'X% Leakage to Outside, R-N'  ou  'X% Leakage to Outside, Uninsulated'  ou  'None'
if 'in.duct_leakage_and_insulation' in df_enc.columns:
    raw = df_enc["in.duct_leakage_and_insulation"].astype(str)

    # Pourcentage de fuite (0 si None)
    df_enc["duct_leakage_pct"] = (
        raw.str.extract(r'^(\d+)%').astype(float).fillna(0).astype('float32')
    )

    # R-value d isolation (0 = Uninsulated ou None)
    df_enc["duct_insulation_r"] = (
        raw.str.extract(r'R-(\d+)').astype(float).fillna(0).astype('float32')
    )

    df_enc.drop(columns=['in.duct_leakage_and_insulation'], inplace=True)
    print('in.duct_leakage_and_insulation -> duct_leakage_pct + duct_insulation_r')

# Vérification
remaining_str = [c for c in df_enc.columns
                 if str(df_enc[c].dtype) in ('object', 'string', 'str')]
print(f'Colonnes string restantes : {remaining_str}')


Colonnes string restantes : ['out.params.panel_constraint_overall.2023_nec_existing_dwelling_load_based']


In [22]:
from pandas.api.types import is_extension_array_dtype

# Colonnes non numériques (object) — vrais strings
string_cols = df_enc.select_dtypes(include=['object']).columns.tolist()

# Colonnes extension pandas (Int8, Int16...) — nullables, pas reconnues par np.number
ext_cols = [c for c in df_enc.columns
            if is_extension_array_dtype(df_enc[c]) and c not in string_cols]

print(f'Colonnes object/string ({len(string_cols)}) :')
for c in string_cols:
    print(f'  {c} — nunique={df_enc[c].nunique()}  ex: {df_enc[c].dropna().unique()[:3].tolist()}')

print(f'\nColonnes extension nullable ({len(ext_cols)}) :')
for c in ext_cols:
    print(f'  {c} — dtype={df_enc[c].dtype}')

if not string_cols and not ext_cols:
    print('Toutes les colonnes sont numeriques.')


C:\Users\bamdyoun\AppData\Local\Temp\claude\ipykernel_25956\1073697658.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df_enc.select_dtypes(include=['object']).columns.tolist()


Colonnes object/string (1) :
  out.params.panel_constraint_overall.2023_nec_existing_dwelling_load_based — nunique=2  ex: ['No Constraint', 'Capacity Constrained Only']

Colonnes extension nullable (19) :
  in.aiannh_area — dtype=Int8
  in.cooling_setpoint_has_offset — dtype=Int8
  in.corridor — dtype=Int8
  in.electric_vehicle_charger — dtype=Int8
  in.electric_vehicle_outlet_access — dtype=Int8
  in.electric_vehicle_ownership — dtype=Int8
  in.geometry_attic_type — dtype=Int8
  in.geometry_building_level_mf — dtype=Int8
  in.geometry_foundation_type — dtype=Int8
  in.geometry_garage — dtype=Int8
  in.has_pv — dtype=Int8
  in.heating_setpoint_has_offset — dtype=Int8
  in.household_has_tribal_persons — dtype=Int8
  in.hvac_has_ducts — dtype=Int8
  in.hvac_has_zonal_electric_heating — dtype=Int8
  in.tenure — dtype=Int8
  in.usage_level — dtype=Int8
  in.vacancy_status — dtype=Int8
  in.water_heater_in_unit — dtype=Int8


In [23]:
out_path = DATA_PROCESSED / 'features_encodees.parquet'
df_enc.to_parquet(out_path, index=False)

print(f'Sauvegardé : {out_path}')
print(f'Shape final : {df_enc.shape}')

Sauvegardé : C:\Users\bamdyoun\stage\FlexiMax\data\processed\features_encodees.parquet
Shape final : (549971, 422)
